## Parte 2. Crear el dashboard

Vamos a echarle otro vistazo al borrador:

![](https://practicum-content.s3.us-west-1.amazonaws.com/resources/moved_11.5.3ES_1655899618.png)





El filtro de fecha y hora y el de país deberán modificar todos los gráficos del dashboard. Fíjate que los gráficos de historial de interacción "Historial de tendencias" = “Historial de tendencias, %” deben tener fecha y hora en el eje X.

“Historial de tendencias” deberá tener el número de videos en la sección de tendencias (el campo `videos_count`) en el eje Y y el otro gráfico deberá tener el porcentaje.

---

Para crear el dashboard, sigue los siguientes pasos:

1. En Tableau Public, usa `trending_by_time.csv` (va al final de esta página para descargarla) para crear un dashboard modelado en el borrador.
2. Publica el dashboard en el servidor de Tableau Public. Asegúrate de que todos puedan acceder a él, e inténtalo abrir en varios navegadores. Si no se puede, el revisor no podrá hacer la revisión.
3. Usa tu dashboard para responder a las preguntas que te hicieron:
   - ¿Qué categorías de videos estuvieron en tendencia más frecuentemente?
   - ¿Cómo se distribuyeron en las regiones?
   - ¿Qué categorías fueron particularmente populares en los Estados Unidos?
   - ¿Hubo diferencias entre las categorías populares en Estados Unidos y en otros lugares?

Prepara una presentación breve con un informe (las respuestas a estas preguntas y gráficas).

Descarga el archivo aquí:  
[**`trending_by_time.csv`**](https://practicum-content.s3.us-west-1.amazonaws.com/datasets/trending_by_time.csv)


In [1]:
import pandas as pd
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px

In [2]:
trending_by_time = pd.read_csv("https://practicum-content.s3.us-west-1.amazonaws.com/datasets/trending_by_time.csv")

In [3]:
trending_by_time.head()

,record_id,region,trending_date,category_title,videos_count
0,1,France,2017-11-14 00:00:00.000000,Autos & Vehicles,8
1,2,France,2017-11-15 00:00:00.000000,Autos & Vehicles,2
2,3,France,2017-11-16 00:00:00.000000,Autos & Vehicles,6
3,4,France,2017-11-17 00:00:00.000000,Autos & Vehicles,8
4,5,France,2017-11-18 00:00:00.000000,Autos & Vehicles,4


In [4]:
# Inicializar la app Dash
app = dash.Dash(__name__)

# Layout
app.layout = html.Div([
    html.H1("Dashboard de Videos en Tendencia"),

    html.Div([
        html.Label("Selecciona una región:"),
        dcc.Dropdown(
            id="region-dropdown",
            options=[{"label": r, "value": r} for r in trending_by_time["region"].unique()],
            value=trending_by_time["region"].unique()[0]
        ),
    ], style={'width': '30%', 'display': 'inline-block'}),

    html.Div([
        html.Label("Selecciona una categoría:"),
        dcc.Dropdown(
            id="category-dropdown",
        ),
    ], style={'width': '30%', 'display': 'inline-block', 'marginLeft': '2%'}),

    dcc.Graph(id="line-chart"),
    dcc.Graph(id="bar-chart")
])

# Callback para actualizar categorías según la región
@app.callback(
    Output("category-dropdown", "options"),
    Output("category-dropdown", "value"),
    Input("region-dropdown", "value")
)
def update_categories(region):
    filtered = trending_by_time[trending_by_time["region"] == region]
    categories = filtered["category_title"].unique()
    options = [{"label": c, "value": c} for c in categories]
    return options, categories[0]

# Callback para actualizar gráficos
@app.callback(
    Output("line-chart", "figure"),
    Output("bar-chart", "figure"),
    Input("region-dropdown", "value"),
    Input("category-dropdown", "value")
)
def update_graphs(region, category):
    filtered = trending_by_time[(trending_by_time["region"] == region) & (trending_by_time["category_title"] == category)]
    line_fig = px.line(
        filtered,
        x="trending_date",
        y="videos_count",
        title=f"Videos en tendencia en '{category}' ({region})",
        markers=True
    )

    bar_data = trending_by_time[trending_by_time["region"] == region]
    bar_fig = px.bar(
        bar_data.groupby("category_title")["videos_count"].sum().reset_index(),
        x="category_title",
        y="videos_count",
        title=f"Total de videos por categoría en {region}"
    )

    return line_fig, bar_fig


In [7]:
app.run(mode='external')